## chatbot Evaluation

In [12]:
## chatbot Evaluation
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["LANGSMITH_API_KEY"] = os.getenv("LANGSMITH_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"] = "true"

from langsmith import Client
client = Client()

dataset_name = "Simple Chatbot Evaluation"

## Safer: try to read it first, only create if it truly doesn't exist
try:
    dataset = client.read_dataset(dataset_name=dataset_name)
    print(f"Found existing dataset: {dataset.id}")
except Exception:
    dataset = client.create_dataset(dataset_name)
    print(f"Created new dataset: {dataset.id}")

## Add examples (safe to re-run; LangSmith dedupes exact same examples on some SDK versions,
## but to be fully safe, check example count first)
existing = list(client.list_examples(dataset_id=dataset.id))
if len(existing) == 0:
    client.create_examples(
        dataset_id=dataset.id,
        inputs=[
            {"question": "What is Langchain?"},
            {"question": "What is langsmith"},
        ],
        outputs=[
            {"answer": "A framework for building llm applications"},
            {"answer": "a platform for observing and evaluation LLM applications"},
        ],
    )
    print("Added 2 examples")
else:
    print(f"Dataset already has {len(existing)} examples, skipping insert")

Found existing dataset: 8ec49d4e-c81c-4b00-86a3-572229b1cda0
Dataset already has 2 examples, skipping insert


### Define Metric (LLM as a judge)

In [13]:

from groq import Groq

groq_client = Groq()

instructions = "You are an exper professor in grading students answer to questions"

def correctness(inputs:dict,outputs:dict,reference_outputs:dict) -> bool:
    user_content = f""" You are grading the following questions:
    {inputs['question']}
Here is the real answer:
{reference_outputs['answer']}
You are grading the following predicted answer:
{outputs['response']}
Respond with CORRECT or INCORRECT:
Grade:
"""
    response = groq_client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "system", "content": instructions},
        {"role": "user", "content": user_content},
    ],
    temperature=0,
  ).choices[0].message.content
   
    return response == "CORRECT"



In [14]:
## Concisions

def concision(outputs:dict,reference_outputs:dict)->bool:
    return int(len(outputs["response"]) < 2 * len(reference_outputs["answer"]))

### how to run evaluation


In [15]:
default_instruction = "Respond to the users question in a short, concise manner(one short sentence)."
def my_app(question:str,model:str="llama-3.3-70b-versatile",instructions:str = default_instruction)->str:
    return groq_client.chat.completions.create(
        model = model,
        temperature=0,
        messages=[
            {"role":"system","content":instructions},
            {"role":"user","content":question},
        ]
    ).choices[0].message.content

In [16]:
### Call my_app for every datapoints
def ls_target(inputs:str)->dict:
    return {"response":my_app(inputs["question"])}

In [17]:
## Run our evaluation

experiment_results = client.evaluate(
    ls_target,
    data=dataset_name,
    evaluators=[correctness,concision],
    experiment_prefix="llama-3.3-70b-versatile"
)

/Users/shivanshmishra2701gmail.com/Desktop/AgenticWorkspace/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


View the evaluation results for experiment: 'llama-3.3-70b-versatile-08b71724' at:
https://smith.langchain.com/o/fc3e80ad-6b48-4352-bc8b-a2ae0f36dfef/datasets/8ec49d4e-c81c-4b00-86a3-572229b1cda0/compare?selectedSessions=4d0ad29f-f6a6-4590-9164-f575c09a1621




2it [00:02,  1.24s/it]
